# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [2]:
import os

repo_name = "2026-HYU-AUE8088-PA2"
repo_path = f"/content/{repo_name}"
repo_url = "https://github.com/Jieunn-Kim/2026-HYU-AUE8088-PA2.git"

if not os.path.exists(repo_path):
    !git clone {repo_url} {repo_path}

%cd {repo_path}

Cloning into '/content/2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 44 (delta 8), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 57.32 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/2026-HYU-AUE8088-PA2


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [6]:
import wandb; wandb.login()   # API key 입력

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jieunnkim (jieunnkim-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [8]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=0e3ad8dc-6ef6-4216-897f-d8dd52db0bcb
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 209MB/s]


압축 해제 중...
완료 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [15]:
from torch import nn

def train_one(model_fn, name, epochs=20):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, f"level1_{name}.pth")

    torch.save(
        {
            "state_dict": model.state_dict(),
            "history": history,
        },
        save_path,
    )

    print("저장 완료:", save_path)

    return model, history

In [14]:
# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)

[epoch 01/30] train_loss=4.1416  val_avg_MF1=0.3733  per={'weather': 0.18901117909511034, 'scene': 0.31062435723825343, 'timeofday': 0.6202562222563245}


[epoch 02/30] train_loss=2.2945  val_avg_MF1=0.4360  per={'weather': 0.20518326254522576, 'scene': 0.3415954415954416, 'timeofday': 0.761109281903587}


[epoch 03/30] train_loss=2.2105  val_avg_MF1=0.4079  per={'weather': 0.20343672637250618, 'scene': 0.37765099124020285, 'timeofday': 0.6424743892828999}


[epoch 04/30] train_loss=2.1359  val_avg_MF1=0.4188  per={'weather': 0.1663596714092141, 'scene': 0.4009551928923402, 'timeofday': 0.6890058474393839}


[epoch 05/30] train_loss=2.0969  val_avg_MF1=0.4357  per={'weather': 0.21853988826674497, 'scene': 0.40241338492252793, 'timeofday': 0.6862094537799542}


...

[epoch 28/30] train_loss=1.5988  val_avg_MF1=0.5511  per={'weather': 0.39691321679394753, 'scene': 0.452005956456995, 'timeofday': 0.8042930527751233}


[epoch 29/30] train_loss=1.6039  val_avg_MF1=0.5394  per={'weather': 0.40716442933021096, 'scene': 0.45233589701027316, 'timeofday': 0.7586651379334306}


[epoch 30/30] train_loss=1.5907  val_avg_MF1=0.5620  per={'weather': 0.4080231069005717, 'scene': 0.4574439218523878, 'timeofday': 0.8205816543499793}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▃▂▃▃▂▃▅▃▄▄▄▅▅▆▅▇▆▆▆▇█▇▇████▇█
val/mf1_scene,▁▂▄▅▅▃▄▆▄▄▄▃█▇▆▅▇▅▇▅█▆▇▇█▇▇▇▇▇
val/mf1_timeofday,▁▆▂▃▃▂▃▅▃▆▅▅▆▅█▅▇▇▇▅▆█▆█▇▇▇▇▆█
val/mf1_weather,▂▂▂▁▃▂▂▃▃▃▃▃▃▄▅▆▆▅▅▇▇▇█▇██████
epoch,30
lr,0
train/loss,1.59067
val/avg_macro_f1,0.56202


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level1_vgg16.pth


In [ ]:
r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)

[epoch 01/30] train_loss=2.1075  val_avg_MF1=0.4805  per={'weather': 0.26085317821922527, 'scene': 0.4348755384564796, 'timeofday': 0.7457805696563726}


[epoch 02/30] train_loss=1.8993  val_avg_MF1=0.4234  per={'weather': 0.2277856799842333, 'scene': 0.33337466739001265, 'timeofday': 0.7091408841034029}


[epoch 03/30] train_loss=1.8359  val_avg_MF1=0.4469  per={'weather': 0.1930647881559263, 'scene': 0.36995299900017753, 'timeofday': 0.7775580258701039}


[epoch 04/30] train_loss=1.7923  val_avg_MF1=0.4933  per={'weather': 0.2576645202489988, 'scene': 0.4233761283503994, 'timeofday': 0.7987632300937021}


[epoch 05/30] train_loss=1.7323  val_avg_MF1=0.4934  per={'weather': 0.3860097169797831, 'scene': 0.36992650825550694, 'timeofday': 0.7243031316836624}


...

[epoch 28/30] train_loss=1.0363  val_avg_MF1=0.6501  per={'weather': 0.5115986951240594, 'scene': 0.6276489493978283, 'timeofday': 0.8110042735042735}


[epoch 29/30] train_loss=1.0359  val_avg_MF1=0.6437  per={'weather': 0.5014027575580359, 'scene': 0.6146923757034778, 'timeofday': 0.8150027855910209}


[epoch 30/30] train_loss=1.0221  val_avg_MF1=0.6444  per={'weather': 0.4973228938154663, 'scene': 0.6106322335933431, 'timeofday': 0.8253024411935738}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁
val/avg_macro_f1,▃▁▂▃▃▃▄▃▅▃▅▅▅▅▅▆▆▆▆▇▇▆▆█▇▇████
val/mf1_scene,▃▁▂▃▂▂▂▁▃▂▅▃▄▃▃▅▇▅▇▇▇▅▆█▇▇▇███
val/mf1_timeofday,▃▁▅▆▂▃▅▅▅▄▆▆▆▆▅▇▆▄▃▆▇▆██▇▆█▇▇█
val/mf1_weather,▂▂▁▂▅▅▆▄▆▅▆▆▆▇▆█▅▇▆█▇▇▆███████
epoch,30
lr,0
train/loss,1.02208
val/avg_macro_f1,0.64442


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level1_resnet18.pth


In [16]:
r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.4377  val_avg_MF1=0.4182  per={'weather': 0.228331253360931, 'scene': 0.3945657094277544, 'timeofday': 0.6318252004145779}


[epoch 02/30] train_loss=2.0562  val_avg_MF1=0.4175  per={'weather': 0.13169239326936447, 'scene': 0.37127672877328993, 'timeofday': 0.7493915016410124}


[epoch 03/30] train_loss=1.9760  val_avg_MF1=0.4835  per={'weather': 0.21555815517970148, 'scene': 0.43924885806124175, 'timeofday': 0.7955890342569344}


[epoch 04/30] train_loss=1.8772  val_avg_MF1=0.3383  per={'weather': 0.2915029668857468, 'scene': 0.3521583167540845, 'timeofday': 0.3713788806542124}


...

[epoch 28/30] train_loss=1.1122  val_avg_MF1=0.6289  per={'weather': 0.5016517957100509, 'scene': 0.5770252042621059, 'timeofday': 0.8080850707226701}


[epoch 29/30] train_loss=1.0778  val_avg_MF1=0.6309  per={'weather': 0.5113012123206998, 'scene': 0.5734574603297703, 'timeofday': 0.8080850707226701}


[epoch 30/30] train_loss=1.0779  val_avg_MF1=0.6250  per={'weather': 0.5105002199249961, 'scene': 0.5543904976437094, 'timeofday': 0.8100848542393301}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val/avg_macro_f1,▃▃▄▁▄▄▁▅▅▆▅▆▇▆▅▆▇▆▇▆██▇█▇▇████
val/mf1_scene,▄▃▅▃▂▄▁▄▄▅▅▄▆▅▄▄▅▃▅▅██▆██▇███▇
val/mf1_timeofday,▅▇▇▁▇▇▂██▇▇██▇▇▇███▇██▇███████
val/mf1_weather,▃▁▃▄▄▂▃▄▄▆▄▅▇▅▄▇▇▇▇▇█▇██▇█████
epoch,30
lr,0
train/loss,1.07788
val/avg_macro_f1,0.62499


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level1_resnet50.pth


In [17]:
# Loss 가중치 민감도: ResNet-18 (2, 1, 1)) 으로 재학습

from torch import nn
import os
import torch


def train_one_weighted(
    model_fn,
    name,
    loss_weights,
    epochs=30,
):
    set_seed(SEED, deterministic=True)

    model = model_fn().to(device)

    optim = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=5e-4,
    )

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim,
        T_max=epochs,
    )

    losses = {
        a: nn.CrossEntropyLoss()
        for a in ATTRIBUTES
    }

    cfg = TrainConfig(epochs=epochs)
    cfg.loss_weights = loss_weights

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name,
            "epochs": epochs,
            "batch": BATCH,
            "lr": 3e-4,
            "weight_decay": 5e-4,
            "seed": SEED,
            "loss_weights": loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(
        model,
        optim,
        sched,
        losses,
        device,
        cfg,
        logger=logger,
    )

    history = trainer.fit(
        train_loader,
        val_loader,
    )

    val_pred, _, val_tgt, _ = collect_predictions(
        model,
        val_loader,
        device,
    )

    cms = confusion_matrices(
        val_pred,
        val_tgt,
    )

    for a in ATTRIBUTES:
        logger.log_confusion_matrix(
            f"final/cm_{a}",
            cms[a],
            CLASS_NAMES[a],
        )

    logger.finish()

    save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(
        save_dir,
        f"level1_{name}.pth",
    )

    torch.save(
        {
            "state_dict": model.state_dict(),
            "history": history,
            "loss_weights": loss_weights,
        },
        save_path,
    )

    print("저장 완료:", save_path)

    return model, history

In [18]:
r18_w211_model, r18_w211_hist = train_one_weighted(
    resnet18,
    "resnet18_w211",
    loss_weights={
        "weather": 2.0,
        "scene": 1.0,
        "timeofday": 1.0,
    },
    epochs=30,
)

[epoch 01/30] train_loss=3.0394  val_avg_MF1=0.4070  per={'weather': 0.2098334859907466, 'scene': 0.2906995735414209, 'timeofday': 0.720611278460348}


[epoch 02/30] train_loss=2.7382  val_avg_MF1=0.4442  per={'weather': 0.23115226874193875, 'scene': 0.3645925312729949, 'timeofday': 0.7368102237583689}


[epoch 03/30] train_loss=2.6418  val_avg_MF1=0.4890  per={'weather': 0.25702379852399626, 'scene': 0.4270545066929053, 'timeofday': 0.782987838291734}


[epoch 04/30] train_loss=2.5735  val_avg_MF1=0.3853  per={'weather': 0.26632279867673503, 'scene': 0.28926644825296477, 'timeofday': 0.6004437617340843}


[epoch 05/30] train_loss=2.4790  val_avg_MF1=0.4676  per={'weather': 0.2771017197186188, 'scene': 0.3848394175144623, 'timeofday': 0.7407464496542885}


...

[epoch 28/30] train_loss=1.3989  val_avg_MF1=0.6320  per={'weather': 0.5291881607155328, 'scene': 0.5739989968602243, 'timeofday': 0.7926855600539812}


[epoch 29/30] train_loss=1.4004  val_avg_MF1=0.6298  per={'weather': 0.5165491600427057, 'scene': 0.5801454327810619, 'timeofday': 0.7926855600539812}


[epoch 30/30] train_loss=1.4059  val_avg_MF1=0.6263  per={'weather': 0.5207201004156358, 'scene': 0.575455398373327, 'timeofday': 0.7828413416648711}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val/avg_macro_f1,▂▃▄▁▃▄▆▅▄▅▆▇▆▇▆▆▆▆▇▇█▇▆▇▇█████
val/mf1_scene,▁▃▄▁▃▃▅▃▃▃▆▆▆▅▆▆▄▆▆▅▇▇▅▆▆▇████
val/mf1_timeofday,▄▅▆▁▅▆▆▅▃▅▇█▆▇▆▇▆▆▇▇▆▇▇█▆▆▆▆▆▆
val/mf1_weather,▁▁▂▂▂▃▆▆▅▇▅▆▆▆▆▅▆▆█▇█▆▆▆▇▇████
epoch,30
lr,0
train/loss,1.40587
val/avg_macro_f1,0.62634


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level1_resnet18_w211.pth


## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.